In [1]:
import os, glob, time, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.metrics import roc_curve

# ==========================================
# 🌱 Seed 고정 (재현성 확보)
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(42)

In [2]:
# ==========================================
# 1. Config & Evaluation Setup
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FS = 128
WINDOW_SIZE = 128 * 4     # 4초
STRIDE = WINDOW_SIZE // 2 # 50% Overlap 

EMBED_DIM = 192
# 🔥 Gating 모델에 맞춘 Dropout (0.45)
DROPOUT_RATE = 0.45

DATA_ROOT = "/Data/CRS25/PPG_Certifiation/data/Final_Data"

# 🌟 평가(OOB)를 진행할 유저를 설정하세요 (1~4번)
OOB_USERS = [13,14,15,16] 
MIN_TO_SAMPLES = 60 * FS

# ==========================================
# 🕒 절대 타임라인 설정 (OOB Test Protocol)
# ==========================================
JUNGFRAU_START = 240  

ACT1_START = JUNGFRAU_START + 0    # 240m (Museum)
ACT2_START = JUNGFRAU_START + 60   # 300m (Elevator)
ACT3_START = JUNGFRAU_START + 120  # 360m (Lunch)
ACT4_START = JUNGFRAU_START + 180  # 420m (Snow walk)
ACT5_START = JUNGFRAU_START + 240  # 480m (Indoor rest)

# 🎯 Enrollment (등록): 실내 휴식 안정기 (5분~10분 구간)
ENROLL_START = ACT5_START + 5      # 485m
ENROLL_END = ACT5_START + 10       # 490m

# 🎯 OOB Test (평가): 각 활동의 중간 30분 (15분~45분 구간)
TEST_PHASES_OOB = [
    (ACT1_START + 15, ACT1_START + 45, "Museum/Ice Cave"), 
    (ACT2_START + 15, ACT2_START + 45, "Elevator/Obs."),   
    (ACT3_START + 15, ACT3_START + 45, "Lunch"),           
    (ACT4_START + 15, ACT4_START + 45, "Snow Walk"),       
    (ACT5_START + 15, ACT5_START + 45, "Indoor Rest"),     
]

In [3]:
# ==========================================
# 2. Model Components (Context-Driven Bounded Gating)
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(channels, channels // reduction), nn.ReLU(inplace=True), nn.Linear(channels // reduction, channels), nn.Sigmoid())
    def forward(self, x): return x * self.fc(self.avg_pool(x).view(x.size(0), x.size(1))).view(x.size(0), x.size(1), 1)

class TDNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, padding=None):
        super().__init__()
        if padding is None: padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels)
    def forward(self, x): return self.bn(F.relu(self.conv(x)))

class Res2NetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, scale=8, dilation=1):
        super().__init__()
        self.width = out_channels // scale
        self.scale = scale
        self.convs = nn.ModuleList([nn.Conv1d(self.width, self.width, 3, dilation=dilation, padding=dilation) for _ in range(scale - 1)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(self.width) for _ in range(scale - 1)])
        self.se = SEBlock(out_channels)
    def forward(self, x):
        chunks = torch.split(x, self.width, dim=1)
        y, out = chunks[0], [chunks[0]]
        for i in range(self.scale - 1):
            y = F.relu(self.bns[i](self.convs[i](chunks[i+1] + (y if i > 0 else 0))))
            out.append(y)
        return self.se(torch.cat(out, dim=1)) + x

class AttentiveStatisticsPooling(nn.Module):
    def __init__(self, channels, attention_channels=128):
        super().__init__()
        self.tdnn = nn.Conv1d(channels, attention_channels, 1)
        self.conv = nn.Conv1d(attention_channels, channels, 1)
    def forward(self, x):
        w = F.softmax(self.conv(torch.tanh(self.tdnn(x))), dim=2)
        mu = torch.sum(x * w, dim=2)
        var = torch.clamp(torch.sum((x**2) * w, dim=2) - mu**2, min=1e-7)
        return torch.cat((mu, torch.sqrt(var)), dim=1)

class ECAPA_TDNN_1D(nn.Module):
    def __init__(self, in_channels, channels=[256, 256, 256, 256, 768], lin_neurons=192, dropout_p=0.45):
        super().__init__()
        self.layer1 = TDNNBlock(in_channels, channels[0], 5, 1)
        self.layer2 = Res2NetBlock(channels[0], channels[1], dilation=2)
        self.layer3 = Res2NetBlock(channels[1], channels[2], dilation=3)
        self.layer4 = Res2NetBlock(channels[2], channels[3], dilation=4)
        self.layer5 = TDNNBlock(channels[1]*3, channels[4], 1, 1)
        self.asp = AttentiveStatisticsPooling(channels[4])
        self.bn_asp = nn.BatchNorm1d(channels[4] * 2)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(channels[4] * 2, lin_neurons)
        self.bn_final = nn.BatchNorm1d(lin_neurons)
        
    def forward(self, x):
        x1 = self.layer1(x)
        x5 = self.layer5(torch.cat((self.layer2(x1), self.layer3(self.layer2(x1)), self.layer4(self.layer3(self.layer2(x1)))), dim=1))
        return self.bn_final(self.fc(self.dropout(self.bn_asp(self.asp(x5).unsqueeze(2)).squeeze(2))))

# 🌟 Bounded Gating 핵심 모듈
class PhysicalContextConditioning(nn.Module):
    def __init__(self, context_dim=8): 
        super().__init__()
        self.acc_proj = nn.Sequential(nn.Linear(1, context_dim), nn.ReLU(inplace=True))
        self.tmp_proj = nn.Sequential(nn.Linear(1, context_dim), nn.ReLU(inplace=True))
        self.gate_generator = nn.Sequential(nn.Linear(context_dim * 2, 3), nn.Sigmoid())

    def forward(self, raw_acc, temp):
        acc_noise_level = torch.mean(torch.var(raw_acc, dim=2), dim=1, keepdim=True)
        tmp_level = torch.mean(temp, dim=2) 
        
        ctx_combined = torch.cat([self.acc_proj(acc_noise_level), self.tmp_proj(tmp_level)], dim=1)
        gates = self.gate_generator(ctx_combined) 
        
        # 코사인 붕괴 방지: 게이트 최소값을 0.5로 강제 (Bounded)
        bounded_gates = 0.5 + (0.5 * gates)
        return bounded_gates[:, 0:1], bounded_gates[:, 1:2], bounded_gates[:, 2:3]

class TriModalECAPAModel(nn.Module):
    def __init__(self, num_users=12, embed_dim=192, dropout_p=0.45):
        super().__init__()
        self.ppg_enc = ECAPA_TDNN_1D(1, lin_neurons=embed_dim, dropout_p=dropout_p)
        self.tmp_enc = ECAPA_TDNN_1D(1, lin_neurons=embed_dim, dropout_p=dropout_p)
        self.acc_enc = ECAPA_TDNN_1D(3, lin_neurons=embed_dim, dropout_p=dropout_p)
        self.context_cond = PhysicalContextConditioning(context_dim=8)
        
        # Concat 방식이므로 Projector 입력은 순정(192*3)과 동일
        self.projector = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), 
            nn.BatchNorm1d(embed_dim),
            nn.Dropout(p=dropout_p)
        )
        
    def forward(self, ppg, temp, acc, return_features=False):
        p_emb, t_emb, a_emb = self.ppg_enc(ppg), self.tmp_enc(temp), self.acc_enc(acc)
        w_p, w_t, w_a = self.context_cond(acc, temp)
        
        # 환경 정보(맥락)를 임베딩에 직접 섞지 않고, 볼륨(가중치)으로만 곱해줌
        e = torch.cat([p_emb * w_p, t_emb * w_t, a_emb * w_a], dim=1)
        final_emb = self.projector(e)
        
        if return_features: return final_emb, p_emb, a_emb 
        return final_emb

In [4]:
# ==========================================
# 3. Helper Functions & Strict OOB Logic
# ==========================================
def load_and_preprocess_user(filepath, fs=128, user_id=None):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Cannot find data file: {filepath}")

    df = pd.read_csv(filepath)
    df.columns = [c.strip() for c in df.columns]

    raw_ppg = df['PPG'].values
    detrended = signal.detrend(raw_ppg)
    b, a = signal.butter(4, [0.5/(0.5*fs), 8.0/(0.5*fs)], btype='band')
    filtered = signal.filtfilt(b, a, detrended)
    
    # 🌟 User 4, 6의 손실 구간을 먼저 NaN으로 처리하여 오염된 데이터가 정계산에 개입하지 못하게 함
    if user_id == 4:
        filtered[3786939:4194811] = np.nan
        print("      ⚠️ [Data Control] User 4: Bad recording block excluded via NaN mask.")
    elif user_id == 6:
        filtered[4337572:4545544] = np.nan
        print("      ⚠️ [Data Control] User 6: Bad recording block excluded via NaN mask.")
        
    calib_start = ENROLL_START * MIN_TO_SAMPLES
    calib_end = ENROLL_END * MIN_TO_SAMPLES
    
    # NaN이 있을 수 있으므로 nanmean, nanstd 사용
    train_ppg_mean = np.nanmean(filtered[calib_start:calib_end])
    train_ppg_std = np.nanstd(filtered[calib_start:calib_end]) + 1e-6
    ppg = (filtered - train_ppg_mean) / train_ppg_std
    
    temp = (df['temperature'].values - 25.0) / (40.0 - 25.0)
    raw_acc = df[['acc_x', 'acc_y', 'acc_z']].values.T
    
    if user_id == 4:
        temp[3786939:4194811] = np.nan
        raw_acc[:, 3786939:4194811] = np.nan
    elif user_id == 6:
        temp[4337572:4545544] = np.nan
        raw_acc[:, 4337572:4545544] = np.nan

    train_acc_mean = np.nanmean(raw_acc[:, calib_start:calib_end], axis=1, keepdims=True)
    train_acc_std = np.nanstd(raw_acc[:, calib_start:calib_end], axis=1, keepdims=True) + 1e-6
    acc = (raw_acc - train_acc_mean) / train_acc_std
    
    return ppg, temp, acc

# 🌟 평가용 배치 사이즈 512로 확대 (엄청난 속도 향상)
def get_embeddings(model, p_t, t_t, a_t, batch_size=512):
    if len(p_t) == 0: return np.array([])
    embs = []
    with torch.no_grad():
        for i in range(0, len(p_t), batch_size):
            p = p_t[i:i+batch_size].to(DEVICE)
            t = t_t[i:i+batch_size].to(DEVICE)
            a = a_t[i:i+batch_size].to(DEVICE)
            feat = model(p, t, a)
            feat = F.normalize(feat, p=2, dim=1)
            embs.append(feat.cpu().numpy())
    return np.concatenate(embs)

def get_clean_anchor_with_bias_shift(model, ppg, temp, acc, num_windows=5, tuning_epochs=10, lr=0.01):
    drop_samples = 25 * FS
    valid_p = ppg[drop_samples:]
    valid_t = temp[drop_samples:]
    valid_a = acc[:, drop_samples:]
    
    n_win = (len(valid_p) - WINDOW_SIZE) // STRIDE + 1
    candidates = []
    
    for i in range(n_win):
        s = i * STRIDE
        e = s + WINDOW_SIZE
        if np.isnan(valid_p[s:e]).any(): continue
        acc_win = valid_a[:, s:e]
        noise = np.sum(np.var(acc_win, axis=1)) 
        candidates.append((s, e, noise))

    candidates.sort(key=lambda x: x[2])
    best_candidates = candidates[:num_windows]

    p_wins = [valid_p[s:e] for s, e, _ in best_candidates]
    t_wins = [valid_t[s:e] for s, e, _ in best_candidates]
    a_wins = [valid_a[:, s:e] for s, e, _ in best_candidates]

    p_t = torch.tensor(np.array(p_wins), dtype=torch.float32).unsqueeze(1).to(DEVICE)
    t_t = torch.tensor(np.array(t_wins), dtype=torch.float32).unsqueeze(1).to(DEVICE)
    a_t = torch.tensor(np.array(a_wins), dtype=torch.float32).to(DEVICE)

    # ==============================================================
    # 🚀 [Few-Shot] Context Bias Shifting (게이팅 편향 이식) 
    # ==============================================================
    model.train() # 학습 모드 전환
    
    # 1. 딥러닝의 모든 파라미터(가중치)를 꽁꽁 얼립니다 (동결)
    for param in model.parameters():
        param.requires_grad = False
        
    # 2. 오직 맥락 모듈(context_cond) 내부의 '편향(bias)' 파라미터만 학습을 허용합니다.
    # 이를 통해 인코더가 훼손되는 Catastrophic Forgetting을 원천 차단합니다.
    for name, param in model.context_cond.named_parameters():
        if 'bias' in name:
            param.requires_grad = True
            
    # 풀려난 Bias 파라미터만 업데이트하는 전용 Optimizer 생성
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    # 3. 5개의 템플릿이 서로 최대한 응집하도록(Cohesion) Bias 영점 조절
    for _ in range(tuning_epochs):
        optimizer.zero_grad()
        
        # 5개 윈도우 특징 추출
        embs = model(p_t, t_t, a_t) 
        embs = F.normalize(embs, p=2, dim=1)
        
        # 5개 벡터의 평균점을 '가상 앵커'로 설정
        mean_anchor = torch.mean(embs, dim=0, keepdim=True)
        mean_anchor = F.normalize(mean_anchor, p=2, dim=1)
        
        # 각 벡터가 평균점(가상 앵커)과 얼마나 가까운지 코사인 유사도 계산
        sim = torch.sum(embs * mean_anchor, dim=1)
        
        # Loss: 유사도가 1.0(완벽 일치)에 가까워지도록 유도 (분산 최소화)
        loss = 1.0 - torch.mean(sim) 
        
        loss.backward()
        optimizer.step()
        
    # 평가를 위해 학습 허용했던 파라미터를 다시 잠그고 Eval 모드로 복귀
    for param in model.parameters():
        param.requires_grad = False
    model.eval()
    # ==============================================================

    # 최종적으로 개인화(영점 조절)가 완료된 모델로 앵커 추출
    with torch.no_grad():
        final_embs = model(p_t, t_t, a_t)
        final_embs = F.normalize(final_embs, p=2, dim=1).cpu().numpy()
        
    anchor = np.mean(final_embs, axis=0) 
    return anchor / (np.linalg.norm(anchor) + 1e-8)

def extract_test_windows(ppg, temp, acc):
    drop_samples = 25 * FS
    valid_p = ppg[drop_samples:]
    valid_t = temp[drop_samples:]
    valid_a = acc[:, drop_samples:]
    
    n_win = len(valid_p) // WINDOW_SIZE
    p_wins, t_wins, a_wins = [], [], []
    for i in range(n_win):
        s = i * WINDOW_SIZE
        e = s + WINDOW_SIZE
        
        # 🌟 NaN 윈도우 원천 차단
        if np.isnan(valid_p[s:e]).any(): continue
            
        p_wins.append(valid_p[s:e])
        t_wins.append(valid_t[s:e])
        a_wins.append(valid_a[:, s:e])
        
    p_t = torch.tensor(np.array(p_wins), dtype=torch.float32).unsqueeze(1)
    t_t = torch.tensor(np.array(t_wins), dtype=torch.float32).unsqueeze(1)
    a_t = torch.tensor(np.array(a_wins), dtype=torch.float32)
    return p_t, t_t, a_t

def compute_metrics(gen_scores, imp_scores):
    if len(gen_scores) == 0 or len(imp_scores) == 0: return 0.0, 0.0, 0.0, 0.0
    scores = np.concatenate([gen_scores, imp_scores])
    thresholds = np.linspace(scores.min(), scores.max(), 1000)
    
    fars = np.array([np.sum(imp_scores >= t) / len(imp_scores) for t in thresholds])
    frrs = np.array([np.sum(gen_scores < t) / len(gen_scores) for t in thresholds])
    
    idx = np.argmin(np.abs(fars - frrs))
    eer = (fars[idx] + frrs[idx]) / 2
    return eer, fars[idx], frrs[idx], thresholds[idx]

def plot_scenario(scores, labels, threshold, scenario_idx, phase_lengths, phase_names):
    if len(scores) == 0: return
    plt.figure(figsize=(16, 5))
    colors = ['green' if l == 1 else 'red' for l in labels]
    plt.scatter(range(len(scores)), scores, c=colors, s=10, alpha=0.7)
    plt.axhline(y=threshold, color='blue', linestyle='--', label=f'Threshold ({threshold:.4f})')
    
    curr_idx = 0
    for i, length in enumerate(phase_lengths):
        if length == 0: continue
        mid_pt = curr_idx + (length // 2)
        plt.text(mid_pt, 1.05, phase_names[i], ha='center', va='bottom', fontsize=9, 
                 bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
        curr_idx += length
        if i < len(phase_lengths) - 1:
            plt.axvline(x=curr_idx, color='gray', linestyle=':', alpha=0.7)
            
    plt.title(f"Bounded Gating OOB Evaluation - Scenario {scenario_idx}", pad=25)
    plt.xlabel("Window Index (Non-overlapping)")
    plt.ylabel("Cosine Similarity")
    plt.ylim(scores.min() - 0.1, 1.15)
    plt.legend(loc='lower left')
    plt.tight_layout()
    plt.savefig(f"OOB_BoundedGating_Scenario_{scenario_idx}.png", dpi=300)
    plt.close()

In [5]:
# ==========================================
# 4. Multi-Model Open-Set Evaluation
# ==========================================
if __name__ == "__main__":
    
    # --- [1] Unseen Users 데이터 메모리에 캐싱 (한 번만 로드) ---
    print("⏳ Loading OOB User Data into memory...")
    user_data = {}
    for u in OOB_USERS:
        filepath = os.path.join(DATA_ROOT, f"user_{u}_final.csv")
        ppg, temp, acc = load_and_preprocess_user(filepath, fs=FS, user_id=u)
        user_data[u] = {'ppg': ppg, 'temp': temp, 'acc': acc}
        print(f"✅ Loaded OOB User {u} (Total length: {len(ppg)/FS/60:.1f} mins)")

    # 🌟 [초고속 최적화 2] Bounded Gating 모델 폴더 내 자동 탐색
    pt_files = sorted(glob.glob("best_DualGating_dropout_ep*.pt"))
    if not pt_files:
        print("\n⚠️ No .pt files found in the current directory!")
    else:
        print(f"\n🎯 Found {len(pt_files)} model(s) to evaluate. Starting batch evaluation...\n")

    # 기본 모델 구조 선언 (Dropout=0.45 셋팅)
    model = TriModalECAPAModel(num_users=12, embed_dim=EMBED_DIM, dropout_p=DROPOUT_RATE).to(DEVICE)
    
    # 시나리오 동적 구성
    scenarios = []
    for tgt in OOB_USERS:
        imps = [u for u in OOB_USERS if u != tgt]
        scenarios.append({"target": tgt, "imps": imps})

    final_results_summary = {}

    # --- 🔄 모델별 순회 평가 시작 ---
    # --- 🔄 모델별 순회 평가 시작 ---
    for pt_file in pt_files:
        print("*"*80)
        print(f"🚀 Evaluating Model: {pt_file}")
        print("*"*80)
        
        # 모델 가중치 로드
        state_dict = torch.load(pt_file, map_location=DEVICE)
        clean_state_dict = {k: v for k, v in state_dict.items() if 'total_ops' not in k and 'total_params' not in k}
        
        # 🌟 Bias 조절 전, '깨끗한 원본 가중치'를 RAM에 보관합니다!
        base_clean_state = {k: v.clone() for k, v in clean_state_dict.items()}

        overall_eers = []
        start_time = time.time()

        for idx, sc in enumerate(scenarios, 1):
            tgt = sc['target']
            imps = sc['imps']
            
            # 🌟 [매우 중요] 매 시나리오(새로운 타겟)마다 
            # 이전 사람에게 맞춰진 Bias를 버리고 깨끗한 원본 상태로 복구!
            model.load_state_dict(base_clean_state, strict=False)
            
            # --- [Step A] Enrollment ---
            e_s, e_e = int(ENROLL_START * MIN_TO_SAMPLES), int(ENROLL_END * MIN_TO_SAMPLES)
            tgt_e_ppg = user_data[tgt]['ppg'][e_s:e_e]
            tgt_e_tmp = user_data[tgt]['temp'][e_s:e_e]
            tgt_e_acc = user_data[tgt]['acc'][:, e_s:e_e]
            
            # 🚀 여기서 Bias Shifting이 일어납니다! (10 Epoch 튜닝)
            anchor_vec = get_clean_anchor_with_bias_shift(
                model, tgt_e_ppg, tgt_e_tmp, tgt_e_acc, 
                num_windows=50, tuning_epochs=5, lr=0.01
            )
            
            # --- [Step B] Test ---
            scenario_scores, scenario_labels = [], []
            
            for phase_start, phase_end, phase_name in TEST_PHASES_OOB:
                s_idx, e_idx = int(phase_start * MIN_TO_SAMPLES), int(phase_end * MIN_TO_SAMPLES)
                
                # Target 평가 
                p_tgt, t_tgt, a_tgt = extract_test_windows(user_data[tgt]['ppg'][s_idx:e_idx], 
                                                           user_data[tgt]['temp'][s_idx:e_idx], 
                                                           user_data[tgt]['acc'][:, s_idx:e_idx])
                tgt_embs = get_embeddings(model, p_tgt, t_tgt, a_tgt)
                tgt_scores = np.dot(tgt_embs, anchor_vec) if len(tgt_embs) > 0 else np.array([])
                
                # Imposters 평가
                imp_scores = []
                for imp in imps:
                    p_imp, t_imp, a_imp = extract_test_windows(user_data[imp]['ppg'][s_idx:e_idx], 
                                                               user_data[imp]['temp'][s_idx:e_idx], 
                                                               user_data[imp]['acc'][:, s_idx:e_idx])
                    imp_embs = get_embeddings(model, p_imp, t_imp, a_imp)
                    if len(imp_embs) > 0:
                        imp_scores.extend(np.dot(imp_embs, anchor_vec))
                imp_scores = np.array(imp_scores)
                
                if len(tgt_scores) > 0 or len(imp_scores) > 0:
                    all_phase_scores = np.concatenate([tgt_scores, imp_scores])
                    all_phase_labels = np.concatenate([np.ones(len(tgt_scores)), np.zeros(len(imp_scores))])
                    scenario_scores.extend(all_phase_scores)
                    scenario_labels.extend(all_phase_labels)
            
            # --- [Step C] 시나리오 결산 ---
            scenario_scores = np.array(scenario_scores)
            scenario_labels = np.array(scenario_labels)
            
            final_eer, _, _, _ = compute_metrics(scenario_scores[scenario_labels == 1], scenario_scores[scenario_labels == 0])
            overall_eers.append(final_eer)
            
        mean_oob_eer = np.mean(overall_eers) * 100
        eval_time = time.time() - start_time
        final_results_summary[pt_file] = mean_oob_eer
        
        print(f"✅ Finished in {eval_time:.2f}s | 🏆 Final OOB EER: {mean_oob_eer:.2f}%")
        print("-" * 80 + "\n")

    # ==========================================
    # 5. 모든 모델 평가 완료 후 랭킹 보드 출력
    # ==========================================
    print("🎉 All Bounded Gating Models evaluated! Here is the summary board:")
    print("="*60)
    print(f"{'Model Name':<50} | {'OOB EER':<8}")
    print("="*60)
    # EER이 낮은(성능이 좋은) 순서대로 정렬하여 출력
    for pt, eer in sorted(final_results_summary.items(), key=lambda item: item[1]):
        print(f"{pt:<50} | {eer:05.2f}%")
    print("="*60)

⏳ Loading OOB User Data into memory...
✅ Loaded OOB User 13 (Total length: 863.7 mins)
✅ Loaded OOB User 14 (Total length: 871.3 mins)
✅ Loaded OOB User 15 (Total length: 795.2 mins)
✅ Loaded OOB User 16 (Total length: 834.0 mins)

🎯 Found 11 model(s) to evaluate. Starting batch evaluation...

********************************************************************************
🚀 Evaluating Model: best_DualGating_dropout_ep13_eer2.75.pt
********************************************************************************
✅ Finished in 26.82s | 🏆 Final OOB EER: 9.30%
--------------------------------------------------------------------------------

********************************************************************************
🚀 Evaluating Model: best_DualGating_dropout_ep16_eer2.36.pt
********************************************************************************
✅ Finished in 21.38s | 🏆 Final OOB EER: 9.96%
--------------------------------------------------------------------------------

*****